In [1]:
import pandas as pd
from pathlib import Path    
from statsmodels.stats.multitest import multipletests
from scipy import stats
import numpy as np

In [16]:
result_path = Path("../results/downstream_task")
methods_list = ["fw-mrs-temperature-mean", "fw-mrs-temperature", "kmm", "uniform", "psa", "mrs-forest"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_negative_class", "less_positive_class"]
metrics = ["AUROC", "AUPRC"]
bias_strengths = ["0.1"]
# datasets = ["breast_cancer", "folktables_employment", "folktables_income", "hr_analytics", "loan_prediction"]
datasets = ["breast_cancer", "loan_prediction"]

In [111]:
method_pairs = []
for j in range(1, len(methods_list)):
    method_pairs.append((methods_list[0], methods_list[j]))
method_pairs

[('fw-mrs-temperature-mean', 'fw-mrs-temperature'),
 ('fw-mrs-temperature-mean', 'kmm'),
 ('fw-mrs-temperature-mean', 'uniform'),
 ('fw-mrs-temperature-mean', 'psa'),
 ('fw-mrs-temperature-mean', 'mrs-forest')]

In [112]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            for bias_strength in bias_strengths:
                json_directory = result_path / dataset / bias_type /  bias_strength/ method / "classification_results"
                auroc_file = pd.read_json(str(json_directory / "rf_auroc.json"))
                auprc_file = pd.read_json(str(json_directory / "rf_auprc.json"))
                dict_list.append({"Method": method, "Data Set": dataset, "AUROC": auroc_file.values, "AUPRC":auprc_file.values, 
                                  "Bias Type": bias_type, "Bias Strength": bias_strength})
result_df = pd.DataFrame(data=dict_list, )

In [133]:
def corrected_t_test(first_values, second_values):
    differences = first_values - second_values
    mean_differences = np.mean(differences)
    variance_differences = np.var(differences)
    corrected_variance = variance_differences / len(first_values)
    correction_factor = 1.0 + ((2.0 / 3.0) / (1.0 / 3.0))
    np.sqrt(corrected_variance * correction_factor)
    if np.sqrt(corrected_variance * correction_factor) == 0:
        print(mean_differences)
        print(first_values)
        print(second_values)
        return mean_differences
    else:
        return mean_differences / (np.sqrt(corrected_variance * correction_factor))

In [134]:
# p_values = np.zeros(len(datasets)*len(bias_types)*len(method_pairs)*len(bias_strengths) * 2)
p_values = np.zeros(6 * len(bias_types) * len(method_pairs) * len(bias_strengths) * 2)
i = 0
for dataset in datasets:
    for bias_type in bias_types:
            for bias_strength in bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in ("AUROC", "AUPRC"):
                        first_metrics = result_df.loc[(result_df["Method"]==first_method_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        second_metrics = result_df.loc[(result_df["Method"]==second_metric_name) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & (result_df["Data Set"]==dataset)][metric].values[0]
                        t_statistic = corrected_t_test(first_metrics, second_metrics)
                        p_values[i] = stats.t.sf(t_statistic, len(first_metrics-1)) * 2
                        i += 1
corrected_p_values = multipletests(p_values, method="fdr_bh")

-0.12759698473984205
[[0.5596617]]
[[0.68725869]]
-0.06913537570351791
[[0.72064096]]
[[0.78977634]]


In [126]:
i = 0
for dataset in datasets:
    for bias_type in bias_types:
            for bias_strength in bias_strengths:
                for first_method_name, second_metric_name in method_pairs:
                    for metric in ("AUROC", "AUPRC"):
                        print(f"p value for {dataset}, {bias_type}, {bias_strength}, {first_method_name}, {second_metric_name} is: {corrected_p_values[1][i]}")
                        i += 1

p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, fw-mrs-temperature is: 0.5678667934488436
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, fw-mrs-temperature is: 0.42811455813783045
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, kmm is: 0.883376049699051
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, kmm is: 0.5924369770347143
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, uniform is: 0.5678667934488436
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, uniform is: 0.3005687269995021
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, psa is: 0.91331352280772
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, psa is: 0.566013379032828
p value for breast_cancer, less_negative_class, 0.1, fw-mrs-temperature-mean, mrs-forest is: 0.7815837944728262
p value for b